In [1]:
import torch
import PIL
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image 
from tqdm import tqdm 
import os

In [2]:
DATASETS_PATH = '../data/grid_noise/images/' #CHANGE

In [3]:
npy_files = [f for f in os.listdir(DATASETS_PATH) if f.endswith('.npy')]
num_images_to_plot = None
if num_images_to_plot is not None:
    plot_ind = np.random.choice(range(len(npy_files)), num_images_to_plot)
else:
    plot_ind = range(len(npy_files))
data_list = []
for ind in tqdm(plot_ind):
    data = np.load(os.path.join(DATASETS_PATH, npy_files[ind]), allow_pickle=True).item()

    # Convert to PIL Image
    # image = Image.fromarray(data['image'])
    image = data['image'][0]
    # Resize to 128x128
    resized_image = image.resize((128, 128))
    
    # Convert back to NumPy array
    resized_array = np.array(resized_image)
    
    data_list.append((resized_array, data['alpha'], data['beta']))

# Define a function to generate Gaussian targets
def generate_gaussian(alpha, beta, grid_size=(128, 128)):
    # Create a grid of coordinates
    x = np.linspace(0, 1, grid_size[0])
    y = np.linspace(0, 1, grid_size[1])
    xx, yy = np.meshgrid(x, y)
    
    # Compute Gaussian centered at (alpha, beta)
    sigma = 0.01  # Standard deviation (adjust as needed)
    gaussian = np.exp(-((xx - alpha)**2 + (yy - beta)**2) / (2 * sigma**2))
    return gaussian
    
states = np.array([item[0] for item in data_list])  # Shape: Nx128x128x3
labels = np.array([[item[1], item[2]] for item in data_list])  # Shape: Nx2
targets = np.array([generate_gaussian(item[1], item[2]) for item in data_list])  # Shape: NxHxW (Gaussian grid)

100%|██████████| 25000/25000 [08:07<00:00, 51.24it/s]


In [ ]:
states = np.transpose(states,(0,3,1,2))
targets = targets[:,None,:,:]

In [6]:
print(f"states.shape = {states.shape}")  # Should be Nx3x128x128
print(f"labels.shape = {labels.shape}")  # Should be Nx2
print(f"targets.shape = {targets.shape}")  # Should be Nx1x128x128

states.shape = (25000, 3, 128, 128)
labels.shape = (25000, 2)
targets.shape = (25000, 1, 128, 128)


In [ ]:
np.save('../data/states.npy',states) #CHANGE
np.save('../data/labels.npy',labels) #CHANGE
np.save('../data/targets.npy',targets) #CHANGE